In [3]:
import shutil
import os

source = "/kaggle/input/datasets/mahmudulhasanshihan/cleanwise-project/cleanwise"
destination = "/kaggle/working/cleanwise"

if os.path.exists(destination):
    shutil.rmtree(destination)

shutil.copytree(source, destination)

print("CleanWise copied successfully!")

CleanWise copied successfully!


In [6]:
%cd /kaggle/working/cleanwise
!ls

/kaggle/working/cleanwise
artifacts  KAGGLE_NOTEBOOK_GUIDE.md  pyproject.toml  requirements.txt
configs    LICENSE		     README.md	     src
data	   notebooks		     reports	     tests


In [7]:
!pip install -e .

Obtaining file:///kaggle/working/cleanwise
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 72.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 82.8 MB/s eta 0:00:00:00:010:01
  Building editable for cleanwise (pyproject.toml) ... done
  Created wheel for cleanwise: filename=cleanwise-0.1.0-0.editable-py3-none-any.whl size=5257 sha256=7c796b584855ea32939a606d7c9437dfcc1991ee1c6d3fed6fd7a4fdd877f5f4
  Stored in directory: /tmp/pip-ephem-wheel-cache-6up8u7jz/wheels/a7/67/d7/5d8d366f72dfc494fc72050d81b8607fc488072cdf63387289
Successfully built cleanwise


In [8]:
!pytest -q

...........                                                              [100%]
11 passed in 2.38s


In [9]:
!python -m cleanwise.benchmark --quick --levels 0.05

Wrote 16 rows across 4 datasets to data/processed/experiments.csv


In [16]:
import pandas as pd

df = pd.read_csv("data/processed/experiments.csv")

print("Dataset shape:", df.shape)

display(
    df[
        [
            "dataset_id",
            "corruption",
            "corruption_level",
            "intervention",
            "model_name",
            "baseline_macro_f1",
            "intervention_macro_f1",
            "delta_macro_f1",
        ]
    ]
)

Dataset shape: (16, 30)


,dataset_id,corruption,corruption_level,intervention,model_name,baseline_macro_f1,intervention_macro_f1,delta_macro_f1
0,breast_cancer,missingness,0.05,median_imputation,logistic_regression,0.985010,0.985010,0.000000
1,breast_cancer,missingness,0.05,iqr_capping,logistic_regression,0.985010,0.992476,0.007465
2,breast_cancer,outliers,0.05,median_imputation,logistic_regression,0.881786,0.881786,0.000000
3,breast_cancer,outliers,0.05,iqr_capping,logistic_regression,0.881786,0.977427,0.095641
4,wine,missingness,0.05,median_imputation,logistic_regression,1.000000,1.000000,0.000000
5,wine,missingness,0.05,iqr_capping,logistic_regression,1.000000,1.000000,0.000000
6,wine,outliers,0.05,median_imputation,logistic_regression,0.867236,0.867236,0.000000
7,wine,outliers,0.05,iqr_capping,logistic_regression,0.867236,1.000000,0.132764
8,iris,missingness,0.05,median_imputation,logistic_regression,0.922963,0.922963,0.000000
9,iris,missingness,0.05,iqr_capping,logistic_regression,0.922963,0.922963,0.000000


In [17]:
print("Average ΔF1 by intervention:")

display(
    df.groupby("intervention")["delta_macro_f1"]
      .agg(["mean", "median", "min", "max"])
      .sort_values("mean", ascending=False)
)

Average ΔF1 by intervention:


,mean,median,min,max
intervention,,,,
iqr_capping,0.038793,0.004558,-0.004361,0.132764
median_imputation,0.000000,0.000000,0.000000,0.000000


In [18]:
print("Best intervention for each experiment group:")

best = (
    df.sort_values("delta_macro_f1", ascending=False)
      .groupby(
          ["dataset_id", "corruption", "corruption_level", "model_name"],
          as_index=False
      )
      .first()
)

display(
    best[
        [
            "dataset_id",
            "corruption",
            "intervention",
            "delta_macro_f1"
        ]
    ]
)

Best intervention for each experiment group:


,dataset_id,corruption,intervention,delta_macro_f1
0,breast_cancer,missingness,iqr_capping,0.007465
1,breast_cancer,outliers,iqr_capping,0.095641
2,digits,missingness,median_imputation,0.000000
3,digits,outliers,iqr_capping,0.077187
4,iris,missingness,iqr_capping,0.000000
5,iris,outliers,iqr_capping,0.001652
6,wine,missingness,median_imputation,0.000000
7,wine,outliers,iqr_capping,0.132764


In [19]:
!python -m cleanwise.train --model ridge

{
  "meta_model": {
    "model": "ridge",
    "mae": 0.029890969499103366,
    "rmse": 0.03647981334302766,
    "r2": -0.14452928702366097,
    "sign_accuracy": 0.25,
    "top1_accuracy": 0.5,
    "mean_regret": 0.00218070172465415,
    "mean_selected_delta": 0.03641259028657205,
    "n_train_datasets": 3,
    "n_test_datasets": 1,
    "n_train_rows": 12,
    "n_test_rows": 4
  },
  "baselines": [
    {
      "strategy": "problem_based_heuristic",
      "mean_selected_delta": 0.0384054415070864,
      "positive_selection_rate": 0.5
    },
    {
      "strategy": "exhaustive_oracle_upper_bound",
      "mean_selected_delta": 0.039338589271908295,
      "positive_selection_rate": 0.625
    }
  ]
}


In [20]:
import json

with open("artifacts/evaluation_metrics.json") as f:
    metrics = json.load(f)

with open("artifacts/baseline_metrics.json") as f:
    baselines = json.load(f)

print("CLEANWISE")
for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nBASELINES")
for baseline in baselines:
    print(baseline)

CLEANWISE
model: ridge
mae: 0.029890969499103366
rmse: 0.03647981334302766
r2: -0.14452928702366097
sign_accuracy: 0.25
top1_accuracy: 0.5
mean_regret: 0.00218070172465415
mean_selected_delta: 0.03641259028657205
n_train_datasets: 3
n_test_datasets: 1
n_train_rows: 12
n_test_rows: 4

BASELINES
{'strategy': 'problem_based_heuristic', 'mean_selected_delta': 0.0384054415070864, 'positive_selection_rate': 0.5}
{'strategy': 'exhaustive_oracle_upper_bound', 'mean_selected_delta': 0.039338589271908295, 'positive_selection_rate': 0.625}


In [21]:
!python -m cleanwise.report

Wrote reports/generated/experiment_report.md


In [22]:
from IPython.display import Markdown, display

report = open(
    "reports/generated/experiment_report.md"
).read()

display(Markdown(report))

# CleanWise Experiment Report

## Scope
This is an automatically generated starter-benchmark report. It is a pipeline validation,
not evidence of general performance across real-world tabular datasets.

## Dataset-level holdout
- Training datasets: 3
- Held-out datasets: 1
- Training experiment rows: 12
- Held-out experiment rows: 4

## Meta-model results
- MAE: 0.029891
- RMSE: 0.036480
- R²: -0.144529
- Sign accuracy: 0.250
- Top-1 intervention accuracy: 0.500
- Mean recommendation regret: 0.002181
- Mean selected delta macro-F1: 0.036413

## Baselines
- **problem_based_heuristic**: mean selected delta=0.038405, positive-selection rate=0.500
- **exhaustive_oracle_upper_bound**: mean selected delta=0.039339, positive-selection rate=0.625

## Interpretation
The exhaustive oracle is an upper bound because it uses measured outcomes for every candidate.
The heuristic is deployable but simple. CleanWise is useful only if it generalizes to unseen
datasets and improves the recommendation/cost trade-off over such baselines.

## Limitations
The starter benchmark contains only four built-in datasets. Expand the benchmark, profile the
observed corrupted training partition, use repeated dataset-grouped validation, add uncertainty,
and measure end-to-end computational cost before making publication-level claims.
